# Nemotron ColEmbed V2 — Colab smoke test

**Purpose**: Verify the visual retriever loads on Colab GPU and produces sensible multi-vector embeddings for a single PDF page.

**Phase**: 0 (environment setup)

**Expected outcome**: Embedding tensor shape printed, something like `(num_patches, hidden_dim)`, with no CUDA OOM.

---

## What this notebook does
1. Install dependencies (HF transformers, pdf2image, pypdfium2)
2. Authenticate HuggingFace (for gated model download)
3. Load Nemotron ColEmbed V2 (default: 4B variant)
4. Convert one PDF page → image → embedding
5. Print embedding shape and a sanity-check similarity score
6. (Optional) Expose a FastAPI endpoint via ngrok so the local Qdrant pipeline can query it

## Prerequisites
- Colab Pro with **L4** runtime (4B model) / T4 (3B) / A100 (8B)
- HuggingFace token (may need to accept model license on HF page)
- One sample PDF uploaded to `/content/` (or mount Google Drive)

## 1. Install dependencies

In [ ]:
!pip install -q transformers accelerate einops sentencepiece pypdfium2 pdf2image Pillow
!pip install -q pyngrok fastapi uvicorn nest-asyncio  # for optional tunnel step

## 2. HuggingFace authentication

Get your token from https://huggingface.co/settings/tokens. First visit the Nemotron model page (see model table below) and click 'Agree to license' if prompted.

In [ ]:
from huggingface_hub import login
import os

# Option A: interactive
# login()

# Option B: Colab secret (preferred) — add HF_TOKEN under the key icon on left sidebar
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
login(token=os.environ['HF_TOKEN'])

## 3. Load Nemotron ColEmbed V2

**Nemotron ColEmbed V2 family** (as of Feb 2026, ViDoRe V3 ranking):

| Model ID | Params | ViDoRe V3 rank | Colab runtime |
|---|---|---|---|
| `nvidia/llama-nemotron-colembed-vl-3b-v2` | 3B | 6th | T4 OK |
| **`nvidia/nemotron-colembed-vl-4b-v2`** ★ default | 4B | **3rd** | L4 recommended |
| `nvidia/nemotron-colembed-vl-8b-v2` | 8B | **1st** | L4 / A100 |

Default: **4B** — best performance/cost balance on Colab Pro L4.

Swap `MODEL_ID` below:
- **3B** if only T4 is available (Colab free tier OK)
- **8B** if you want the #1-ranked model for blog story (needs L4 minimum)

In [ ]:
import torch
from transformers import AutoModel, AutoProcessor

# ─── Choose one ────────────────────────────────────────
MODEL_ID = 'nvidia/nemotron-colembed-vl-4b-v2'          # ★ default (3rd on ViDoRe V3)
# MODEL_ID = 'nvidia/llama-nemotron-colembed-vl-3b-v2'  # T4-friendly (6th)
# MODEL_ID = 'nvidia/nemotron-colembed-vl-8b-v2'        # SOTA (1st), needs L4+
# ───────────────────────────────────────────────────────

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
).eval()

print(f'Model loaded. Params: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B')

## 4. Sample PDF page → image

Upload `Q1.pdf` (or any Sanofi press release) to `/content/` first, or mount Drive.

In [ ]:
from pdf2image import convert_from_path

PDF_PATH = '/content/Q1.pdf'  # adjust
PAGE_NUM = 1                   # 1-indexed

pages = convert_from_path(PDF_PATH, first_page=PAGE_NUM, last_page=PAGE_NUM, dpi=150)
img = pages[0]
print(f'Page {PAGE_NUM}: {img.size} (w, h)')
img

## 5. Embedding extraction — the actual smoke test

Expected shape for ColBERT-style models: `[1, num_patches, hidden_dim]` — e.g., `[1, 1024, 128]` for a high-res page.

In [ ]:
with torch.no_grad():
    inputs = processor(images=img, return_tensors='pt').to(model.device)
    outputs = model(**inputs)

# The API varies by model; typical attrs:
#   outputs.last_hidden_state  (patch-level, [batch, num_patches, hidden])
#   outputs.embeddings          (some models)
print('Output attrs:', [a for a in dir(outputs) if not a.startswith('_')])

emb = outputs.last_hidden_state if hasattr(outputs, 'last_hidden_state') else outputs.embeddings
print(f'Embedding shape: {emb.shape}')          # expected: [1, num_patches, hidden_dim]
print(f'dtype: {emb.dtype}, device: {emb.device}')

## 6. Sanity: query embedding + MaxSim score

ColBERT-style retrieval uses MaxSim: for each query token, take the max similarity over all document patches, then sum. Mini version here to confirm the math works.

In [ ]:
# Encode a text query (model should support text input too)
query = 'What was Dupixent sales in Q1 2025?'
with torch.no_grad():
    q_inputs = processor(text=query, return_tensors='pt').to(model.device)
    q_out = model(**q_inputs)

q_emb = q_out.last_hidden_state if hasattr(q_out, 'last_hidden_state') else q_out.embeddings
print(f'Query emb shape: {q_emb.shape}')

# MaxSim: [1, Lq, D] x [1, Lp, D] -> [1, Lq, Lp] -> max over Lp -> sum over Lq
import torch.nn.functional as F
q_norm = F.normalize(q_emb, dim=-1)
d_norm = F.normalize(emb, dim=-1)
sim = torch.einsum('bqd,bpd->bqp', q_norm, d_norm)
maxsim = sim.max(dim=-1).values.sum(dim=-1)
print(f'MaxSim score (sanity): {maxsim.item():.3f}')

## 7. (Optional) FastAPI tunnel for local Qdrant pipeline

Wrap the embedding call behind an HTTP endpoint and expose via ngrok so the local indexing script (on your PC) can call it.

Copy the printed URL into `.env` as `COLAB_EMBEDDING_URL`.

In [ ]:
import nest_asyncio, uvicorn
from fastapi import FastAPI, UploadFile, File
from pyngrok import ngrok
from PIL import Image
import io, base64

app = FastAPI()

@app.post('/embed_image')
async def embed_image(file: UploadFile = File(...)):
    img = Image.open(io.BytesIO(await file.read())).convert('RGB')
    with torch.no_grad():
        inputs = processor(images=img, return_tensors='pt').to(model.device)
        out = model(**inputs)
    emb = out.last_hidden_state if hasattr(out, 'last_hidden_state') else out.embeddings
    return {'shape': list(emb.shape), 'embedding_b64': base64.b64encode(emb.cpu().numpy().tobytes()).decode()}

@app.post('/embed_text')
async def embed_text(payload: dict):
    q = payload['query']
    with torch.no_grad():
        inputs = processor(text=q, return_tensors='pt').to(model.device)
        out = model(**inputs)
    emb = out.last_hidden_state if hasattr(out, 'last_hidden_state') else out.embeddings
    return {'shape': list(emb.shape), 'embedding_b64': base64.b64encode(emb.cpu().numpy().tobytes()).decode()}

# Tunnel
# ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))  # optional but recommended
public_url = ngrok.connect(8000)
print(f'COLAB_EMBEDDING_URL={public_url}')

nest_asyncio.apply()
uvicorn.run(app, host='0.0.0.0', port=8000)

## Acceptance checklist (Phase 0)

- [ ] Model loads without OOM
- [ ] Embedding shape printed — should be 3D tensor `[1, L, D]`
- [ ] MaxSim score is a finite positive number (not NaN, not zero)
- [ ] (Optional) ngrok tunnel prints a URL reachable from the host PC

Once all mandatory items pass, Phase 0 is cleared for this component. Move on to Docling + Qdrant setup locally.